In [296]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3
import sys

In [297]:
def create_ford_database(db_name="ford_inventory.db"):
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    
    # Create the vehicles table
    create_table_query = """
    CREATE TABLE IF NOT EXISTS vehicles (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        model TEXT NOT NULL,
        price REAL NOT NULL,
        year INTEGER NOT NULL,
        edition TEXT DEFAULT 'Normal',
        color TEXT NOT NULL,
        country TEXT NOT NULL,
        available_city TEXT NOT NULL,
        quantity_available INTEGER NOT NULL
    )
    """
    
    cursor.execute(create_table_query)
    conn.commit()
    
    print(f"Database '{db_name}' created successfully with 'vehicles' table")
    return conn

def populate_ford_database(conn, inventory_data):
    cursor = conn.cursor()
    
    # Insert inventory data
    insert_query = """
    INSERT INTO vehicles (model, price, year, edition, color, country, available_city, quantity_available)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """
    
    for vehicle_id, vehicle_data in inventory_data.items():
        cursor.execute(insert_query, (
            vehicle_data['model'],
            vehicle_data['price'],
            vehicle_data['year'],
            vehicle_data['edition'],
            vehicle_data['color'],
            vehicle_data['country'],
            vehicle_data['available_city'],
            vehicle_data['quantity_available']
        ))
    
    conn.commit()
    print(f"Inserted {len(inventory_data)} vehicles into the database")


In [298]:
with open('Test_Data/Ford.json', 'r') as f:
    ford_inventory = json.load(f)

for vehicle_id, vehicle_data in ford_inventory.items():
    print(vehicle_data['model'])


Mustang
Mustang
Mustang
Mustang
F-150
F-150
F-150
Explorer
Explorer
Bronco
Bronco
Bronco
Escape
Escape
Maverick
Maverick
Mustang Mach-E
Mustang Mach-E
Mustang Mach-E
Expedition


In [299]:
ford_db = create_ford_database()
cursor = ford_db.cursor()
cursor.execute("SELECT COUNT(*) FROM vehicles")
row_count = cursor.fetchone()[0]
with open('Test_Data/Ford.json', 'r') as f:
    ford_inventory = json.load(f)
if row_count == 0: populate_ford_database(ford_db, ford_inventory)
 
# Get row count

print(f"Total vehicles in database: {row_count}")

Database 'ford_inventory.db' created successfully with 'vehicles' table
Total vehicles in database: 23


In [300]:
def add_vehicle_to_database(conn, model, price, year, color, country, available_city, quantity_available, edition="Normal"):
    cursor = conn.cursor()
    
    insert_query = """
    INSERT OR REPLACE INTO vehicles (model, price, year, edition, color, country, available_city, quantity_available)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """
    
    cursor.execute(insert_query, (
        model,
        price,
        year,
        edition,
        color,
        country,
        available_city,
        quantity_available
    ))
    
    conn.commit()
    print(f"Added/Updated vehicle: {model} {edition} ({color}) - ${price}")

In [301]:
def filter_cars_from_db(conn, model=None, max_price=None, city=None, country=None, 
                         color=None, year=None, edition=None):
    cursor = conn.cursor()
    # Build the query dynamically
    query = "SELECT * FROM vehicles WHERE 1=1"
    params = []
    
    if model:
        query += " AND model = ?"
        params.append(model)
    if max_price:
        query += " AND price <= ?"
        params.append(max_price)
    if city:
        query += " AND available_city = ?"
        params.append(city)
    if country:
        query += " AND country = ?"
        params.append(country)
    if color:
        query += " AND color = ?"
        params.append(color)
    if year:
        query += " AND year = ?"
        params.append(year)
    if edition:
        query += " AND edition = ?"
        params.append(edition)
    
    cursor.execute(query, params)
    rows = cursor.fetchall()
    
    # Convert rows to dictionaries
    columns = [description[0] for description in cursor.description]
    results = []
    for row in rows:
        vehicle_dict = dict(zip(columns, row))
        results.append(vehicle_dict)
    
    return results

In [302]:
insert_function = {
    "name": "add_vehicle_to_database",
    "description": (
        "Add a new vehicle to the Ford inventory database. "
        "Only the model name is required; other fields are optional."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "model": {
                "type": "string",
                "description": "Vehicle model name (required), e.g. Mustang, F-150, Explorer"
            },
            "price": {
                "type": "number",
                "description": "Vehicle price (optional)"
            },
            "year": {
                "type": "integer",
                "description": "Vehicle year (optional)"
            },
            "color": {
                "type": "string",
                "description": "Vehicle color (optional)"
            },
            "country": {
                "type": "string",
                "description": "Country where vehicle is available (optional)"
            },
            "available_city": {
                "type": "string",
                "description": "City where vehicle is available (optional)"
            },
            "quantity_available": {
                "type": "integer",
                "description": "Available quantity (optional)"
            },
            "edition": {
                "type": "string",
                "description": "Vehicle edition (optional), e.g. Normal, GT, Raptor. Defaults to 'Normal' if not provided"
            }
        },
        "required": ["model"],
        "additionalProperties": False
    },
    "strict": False
}

In [303]:
fetch_function = {
    "name": "fetch_vehicles_from_db",
    "description": (
        "Fetch vehicles from the Ford inventory database using optional filters. "
        "Can search by ID, model, price range, location, color, year, edition, or retrieve all vehicles."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "vehicle_id": {
                "type": "integer",
                "description": "Specific vehicle ID to fetch (optional)"
            },
            "model": {
                "type": "string",
                "description": "Vehicle model name, e.g. Mustang, F-150, Explorer (optional)"
            },
            "min_price": {
                "type": "number",
                "description": "Minimum vehicle price (optional)"
            },
            "max_price": {
                "type": "number",
                "description": "Maximum vehicle price (optional)"
            },
            "city": {
                "type": "string",
                "description": "City where vehicle is available (optional)"
            },
            "country": {
                "type": "string",
                "description": "Country where vehicle is available (optional)"
            },
            "color": {
                "type": "string",
                "description": "Vehicle color (optional)"
            },
            "year": {
                "type": "integer",
                "description": "Vehicle year (optional)"
            },
            "edition": {
                "type": "string",
                "description": "Vehicle edition, e.g. Normal, GT, Raptor (optional)"
            },
            "limit": {
                "type": "integer",
                "description": "Maximum number of results to return (optional)"
            }
        },
        "required": [],
        "additionalProperties": False
    },
    "strict": False
}

In [304]:
tools = [{"type": "function", "function": insert_function},
{"type": "function", "function": fetch_function}]

In [305]:
system_message = """You are a helpful Ford vehicle inventory assistant. You have access to a database of Ford vehicles with the following capabilities:

1. Add new vehicles to the inventory (model, price, year, edition, color, country, city, quantity). The `edition` field is optional. All other fields are mandatory. If the user does not provide a value for any required field, ask them for the missing information before accessing or calling any tools.

When users ask about Ford vehicles, use the available tools to query the database or add new vehicles. Be helpful, accurate, and provide clear information about available inventory. If a user requests information that requires database access, use the appropriate tool to fetch the data.

Always format monetary values in USD and provide clear, concise responses."""

In [306]:
load_dotenv(override=True)
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY')

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

anthropic_url = "https://api.anthropic.com/v1/"
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
openai = OpenAI()

anthropic_model = "claude-haiku-4-5-20251001"
openai_model = "gpt-4o-mini"

Anthropic API Key exists and begins sk-ant-
OpenAI API Key exists and begins sk-proj-


In [307]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=openai_model, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        print('tool_call')
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=openai_model, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [308]:
def handle_tool_calls(message):
    tool_responses = []
    if not message.tool_calls:
        return tool_responses
    
    for tool_call in message.tool_calls:
        conn = sqlite3.connect('ford_inventory.db')
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)
        
        # Execute the function with the provided arguments
        if function_name == "add_vehicle_to_database":
            # Add database connection to arguments
            function_args['conn'] = conn
            result = add_vehicle_to_database(**function_args)
        elif function_name == "fetch_vehicles_from_db":
            # Add database connection to arguments
            function_args['conn'] = conn
            result = filter_cars_from_db(**function_args)
        else:
            result = {"error": f"Unknown function: {function_name}"}
        
        # Format the result as a tool response
        tool_response = {
            "tool_call_id": tool_call.id,
            "role": "tool",
            "content": json.dumps(result)
        }
        tool_responses.append(tool_response)
        conn.close()
    
    return tool_responses

In [309]:
# gr.ChatInterface(fn=chat).launch()

In [312]:
conn = sqlite3.connect('ford_inventory.db')
cursor = conn.cursor()
cursor.execute("SELECT * FROM vehicles")
print(cursor.fetchall())

[(1, 'Mustang', 52000.0, 2025, 'Normal', 'Blue', 'USA', 'New York', 5), (2, 'Mustang', 58000.0, 2025, 'Batman', 'Black', 'USA', 'Los Angeles', 2), (3, 'Mustang', 55000.0, 2025, 'Normal', 'Red', 'USA', 'Chicago', 4), (4, 'Mustang', 61000.0, 2024, 'Dark Knight', 'Black', 'Canada', 'Toronto', 1), (5, 'F-150', 48000.0, 2025, 'Normal', 'White', 'USA', 'Houston', 8), (6, 'F-150', 65000.0, 2025, 'Raptor', 'Black', 'USA', 'Dallas', 3), (7, 'F-150', 62000.0, 2024, 'Raptor', 'Blue', 'Canada', 'Calgary', 2), (8, 'Explorer', 45000.0, 2025, 'Normal', 'Silver', 'USA', 'Miami', 6), (9, 'Explorer', 51000.0, 2025, 'ST', 'Red', 'Canada', 'Montreal', 3), (10, 'Bronco', 47000.0, 2025, 'Normal', 'Green', 'USA', 'Denver', 5), (11, 'Bronco', 59000.0, 2025, 'Wildtrak', 'Orange', 'USA', 'Phoenix', 2), (12, 'Bronco', 57000.0, 2024, 'Wildtrak', 'Black', 'Canada', 'Vancouver', 4), (13, 'Escape', 33000.0, 2025, 'Normal', 'White', 'Canada', 'Ottawa', 10), (14, 'Escape', 38000.0, 2025, 'Sport', 'Blue', 'Canada', 'Mo